# 案例三：替海岸照片試做故事書風格

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnychao/python-machine-learning-2026-student/blob/main/notebooks/ai_solution_practicum/03_style_transfer_story.ipynb)

社區想把海岸照片做成故事書頁面。你會使用已訓練好的風格轉換模型，
比較兩種混合強度，再由人決定哪一版兼顧內容與畫風。

**情境問題：** 只把風格混合強度從 0.35 提高到 0.70，
內容相似度與色彩距離如何改變？

- baseline：混合強度 0.35。
- candidate：只把混合強度改成 0.70。
- [Kaggle 題目出處：I’m Something of a Painter Myself]
  (https://www.kaggle.com/competitions/gan-getting-started)

課堂採用輕量的任意風格轉換，不在現場訓練 GAN。兩張練習圖會由
本課程專案直接下載。


## 1. 安裝風格轉換模型套件


In [ ]:
%pip -q install "tensorflow-hub>=0.16,<0.17"


In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

os.environ["TFHUB_MODEL_LOAD_FORMAT"] = "COMPRESSED"
import tensorflow_hub as hub

CONTENT_URL = "https://raw.githubusercontent.com/johnnychao/python-machine-learning-2026-student/main/data/ai_solution_practicum/assets/style-content-coast.png"
STYLE_URL = "https://raw.githubusercontent.com/johnnychao/python-machine-learning-2026-student/main/data/ai_solution_practicum/assets/style-reference-sea-wind.png"
MODEL_URL = "https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2"
BASELINE_STRENGTH = 0.35
CANDIDATE_STRENGTH = 0.70


## 2. Colab 直接下載兩張課程圖片


In [ ]:
content_path = tf.keras.utils.get_file(
    "style-content-coast.png",
    origin=CONTENT_URL,
    cache_dir="/content",
    cache_subdir="style_assets",
)
style_path = tf.keras.utils.get_file(
    "style-reference-sea-wind.png",
    origin=STYLE_URL,
    cache_dir="/content",
    cache_subdir="style_assets",
)


def load_image(path: str, max_dim: int = 512) -> tf.Tensor:
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False,
    )
    image = tf.image.convert_image_dtype(image, tf.float32)
    shape = tf.cast(tf.shape(image)[:-1], tf.float32)
    scale = max_dim / tf.reduce_max(shape)
    new_shape = tf.cast(shape * scale, tf.int32)
    image = tf.image.resize(image, new_shape)
    return image[tf.newaxis, :]


content_image = load_image(content_path)
style_image = load_image(style_path, max_dim=384)

figure, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(content_image[0])
axes[0].set_title("內容圖：海岸")
axes[1].imshow(style_image[0])
axes[1].set_title("參考畫風：海風")
for axis in axes:
    axis.axis("off")
plt.show()


## 3. 產生一次完整風格圖

第一次載入模型會下載模型檔，請耐心等候。模型輸出只做為候選素材，
不等於美感判斷。


In [ ]:
stylization_model = hub.load(MODEL_URL)
stylized_full = stylization_model(
    content_image,
    style_image,
)[0]
stylized_full = tf.clip_by_value(stylized_full, 0.0, 1.0)
print("風格圖尺寸:", stylized_full.shape)


## 4. 只改一項：混合強度

其他圖片、模型與輸出都相同，只改 strength。


In [ ]:
def blend_with_content(strength: float) -> tf.Tensor:
    return tf.clip_by_value(
        (1.0 - strength) * content_image + strength * stylized_full,
        0.0,
        1.0,
    )


baseline_image = blend_with_content(BASELINE_STRENGTH)
candidate_image = blend_with_content(CANDIDATE_STRENGTH)

figure, axes = plt.subplots(1, 3, figsize=(16, 5))
for image, title, axis in zip(
    [content_image, baseline_image, candidate_image],
    [
        "原始內容",
        f"baseline：{BASELINE_STRENGTH:.2f}",
        f"candidate：{CANDIDATE_STRENGTH:.2f}",
    ],
    axes,
):
    axis.imshow(image[0])
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()


## 5. 比較前後證據

SSIM 越高表示越保留原圖結構；色彩距離越大表示平均色彩改變越多。
這兩個數字都不是「美不美」的分數。


In [ ]:
def image_metrics(image: tf.Tensor) -> dict:
    content_ssim = tf.image.ssim(
        content_image,
        image,
        max_val=1.0,
    )
    content_palette = tf.reduce_mean(content_image, axis=[1, 2])
    image_palette = tf.reduce_mean(image, axis=[1, 2])
    palette_distance = tf.reduce_mean(
        tf.abs(content_palette - image_palette)
    )
    return {
        "content_ssim": float(content_ssim.numpy()[0]),
        "palette_distance_from_content": float(palette_distance.numpy()),
    }


baseline_metrics = image_metrics(baseline_image)
candidate_metrics = image_metrics(candidate_image)
comparison = pd.DataFrame(
    [baseline_metrics, candidate_metrics],
    index=["baseline_0.35", "candidate_0.70"],
)
display(comparison.style.format("{:.3f}"))


## 6. 失敗、限制與人工判斷

請放大檢查海岸線、人物輪廓與天空：

- 若細節扭曲、主體消失或出現不自然紋理，就不能只看平均指標。
- SSIM 與色彩距離不能代表品牌一致性、文化適切性或著作權風險。
- 發布前要由人確認圖片來源、使用範圍與是否誤導觀眾。
- 本例只能比較兩個強度，不代表 0.35 或 0.70 永遠最佳。

請寫一句結論：你會選哪一版放入故事書？保留了什麼，又犧牲了什麼？


## 7. 下載兩張候選圖與實驗紀錄


In [ ]:
baseline_path = Path("/content/style_baseline_035.png")
candidate_path = Path("/content/style_candidate_070.png")
tf.keras.utils.save_img(baseline_path, baseline_image[0])
tf.keras.utils.save_img(candidate_path, candidate_image[0])

experiment_record = {
    "case": "style_transfer_story",
    "question": "提高混合強度後，內容相似度與色彩距離如何改變？",
    "baseline": {
        "strength": BASELINE_STRENGTH,
        "metrics": baseline_metrics,
    },
    "candidate": {
        "strength": CANDIDATE_STRENGTH,
        "metrics": candidate_metrics,
    },
    "single_change": "風格混合強度",
    "human_review": "輪廓、品牌一致性、文化適切性、圖片使用範圍",
    "limitation": "代理指標不等於美感或可發布性",
}

record_path = Path("/content/style_transfer_experiment.json")
record_path.write_text(
    json.dumps(experiment_record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(baseline_path, candidate_path, record_path)


In [ ]:
try:
    from google.colab import files
    for output_path in [baseline_path, candidate_path, record_path]:
        files.download(str(output_path))
except ImportError:
    print("檔案已保留在 /content")
